# Grille de paramètres UMAP + HDBSCAN (cellule indépendante)
Cette cellule est **autonome** : elle relit `VigStates_Global.xlsx`, refait sa propre matrice (AW, QW, NREM, REM, normalisée par la somme, comme dans le notebook) et n'utilise que des variables préfixées `grid_`. Elle ne modifie rien du notebook principal : tu peux la lancer seule, dans un autre notebook, ou la coller à la fin de celui-ci.

**Lecture du tableau** : on cherche des paramètres où (1) l'accord entre graines (`ARI_seeds`) est élevé, (2) le nombre de clusters varie peu d'une graine à l'autre, (3) le % de bruit reste raisonnable. `DBCV` est calculé dans l'espace UMAP : à utiliser comme indice relatif, pas comme preuve.


### Load data

In [8]:
import itertools, time, warnings
import numpy as np
import pandas as pd
import umap, hdbscan
from pathlib import Path
from sklearn.metrics import adjusted_rand_score
from sklearn.manifold import trustworthiness
from ipyfilechooser import FileChooser
from IPython.display import display
warnings.filterwarnings("ignore")

In [9]:
try: # tries to retrieve dpath either from a previous run or from a previous notebook
    %store -r dpath
except:
    print("the path was not defined in store")
    #dpath = "/Users/mb/Documents/Syntuitio/AudreyHay/PlanB/ExampleRedLines/2022_08_06/13_30_01/My_V4_Miniscope/"
    dpath = "//10.69.168.1/crnldata/waking/audrey_hay/L1imaging/"

fc1 = FileChooser(dpath,select_default=True, show_only_dirs = True, title = "<b>Folder with videos</b>")
display(fc1)

# Sample callback function
def update_my_folder(chooser):
    global dpath
    dpath = chooser.selected
    %store dpath
    return 

# Register callback function
fc1.register_callback(update_my_folder)

FileChooser(path='/crnldata/forgetting/Théa/MiniscopeOE_analysis/Allocentrick_task/1_VigSt_2026-10-23_17_19_36…

Load Packages

In [10]:


# ------------------ À ADAPTER ------------------
grid_dpath    = dpath                      
grid_filename = "VigStates_Global.xlsx"
grid_NrSubtype = 'TheaM_other'    
grid_DataType  = 'NormalizedAUC_calcium'
grid_states    = ['AW', 'QW', 'NREM', 'REM']      
grid_seeds     = [0, 1, 2, 3, 4]                   # + de graines = + fiable mais + long
grid_n_neighbors      = [15, 30, 50, 90]
grid_min_dist         = [0.0, 0.1, 0.5, 0.9]
grid_min_cluster_size = [10, 15, 20]
grid_min_samples      = [1, 5, 10]
grid_trust_k = 15   # nb  paired with trustworthiness (défaut sklearn = 5)
# Actual parmeters noted "*" in the board 
grid_current = (90, 0.9, 15, 1)   # (n_neighbors, min_dist, min_cluster_size, min_samples)
# -----------------------------------------------

# 1) Données (même logique que le notebook : baseline, un NeuronType, moyenne par neurone, sans IS/undefined, sans NaN)
grid_file = Path(grid_dpath) / grid_filename
assert grid_file.exists(), f"Fichier introuvable : {grid_file}"
print("Fichier lu :", grid_file)
grid_df = pd.read_excel(grid_file, index_col=0)
grid_df['Unit_ID'] = grid_df['Mice'] + grid_df['Unique_Unit'].astype(int).astype(str)
grid_df['NormalizedAUC_calcium'] = grid_df['AUC_calcium'] / grid_df['DurationSubstate']
grid_df = grid_df[(grid_df['Drug'] == 'baseline') & (grid_df['NeuronType'] == grid_NrSubtype)]
grid_X = grid_df.pivot_table(index='Unit_ID', columns='Substate', values=grid_DataType, aggfunc='mean')[grid_states].dropna()
grid_X = grid_X.div(grid_X.sum(axis=1), axis=0)
print(f"{grid_X.shape[0]} neurones, états : {grid_states}")

# 2) UMAP : 1 fit par (n_neighbors, min_dist, graine) ; HDBSCAN testé ensuite sur chaque embedding
grid_rows = []
t0 = time.time()
n_umap = len(grid_n_neighbors) * len(grid_min_dist)
k = 0
for nn, md in itertools.product(grid_n_neighbors, grid_min_dist):
    k += 1
    embeddings = [umap.UMAP(n_neighbors=nn, min_dist=md, n_components=2, random_state=s).fit_transform(grid_X)
                  for s in grid_seeds]
    grid_trust = np.mean([trustworthiness(grid_X.values, E, n_neighbors=grid_trust_k) for E in embeddings])
    for mcs, ms in itertools.product(grid_min_cluster_size, grid_min_samples):
        labs, ncl, noise, dbcv = [], [], [], []
        for E in embeddings:
            h = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=ms, gen_min_span_tree=True).fit(E)
            labs.append(h.labels_)
            ncl.append(len(set(h.labels_)) - (1 if -1 in h.labels_ else 0))
            noise.append(np.mean(h.labels_ == -1) * 100)
            try: dbcv.append(h.relative_validity_)
            except Exception: dbcv.append(np.nan)
        ari = [adjusted_rand_score(labs[i], labs[j]) for i in range(len(labs)) for j in range(i + 1, len(labs))]
        grid_rows.append(dict(
            current='*' if (nn, md, mcs, ms) == grid_current else '',
            n_neighbors=nn, min_dist=md, min_cluster_size=mcs, min_samples=ms,
            ARI_seeds=np.mean(ari), n_clusters_mean=np.mean(ncl), n_clusters_min=min(ncl), n_clusters_max=max(ncl),
            trustworthiness=grid_trust,
            noise_pct=np.mean(noise), DBCV=np.nanmean(dbcv)))
    print(f"UMAP {k}/{n_umap} (n_neighbors={nn}, min_dist={md}) - {time.time()-t0:.0f}s")

grid_results = pd.DataFrame(grid_rows).sort_values('ARI_seeds', ascending=False).reset_index(drop=True)


Fichier lu : /crnldata/forgetting/Théa/MiniscopeOE_analysis/Allocentrick_task/1_VigSt_2026-10-23_17_19_36/VigStates_Global.xlsx
277 neurones, états : ['AW', 'QW', 'NREM', 'REM']
UMAP 1/16 (n_neighbors=15, min_dist=0.0) - 4s
UMAP 2/16 (n_neighbors=15, min_dist=0.1) - 7s
UMAP 3/16 (n_neighbors=15, min_dist=0.5) - 9s
UMAP 4/16 (n_neighbors=15, min_dist=0.9) - 11s
UMAP 5/16 (n_neighbors=30, min_dist=0.0) - 14s
UMAP 6/16 (n_neighbors=30, min_dist=0.1) - 16s
UMAP 7/16 (n_neighbors=30, min_dist=0.5) - 19s
UMAP 8/16 (n_neighbors=30, min_dist=0.9) - 23s
UMAP 9/16 (n_neighbors=50, min_dist=0.0) - 27s
UMAP 10/16 (n_neighbors=50, min_dist=0.1) - 30s
UMAP 11/16 (n_neighbors=50, min_dist=0.5) - 34s
UMAP 12/16 (n_neighbors=50, min_dist=0.9) - 37s
UMAP 13/16 (n_neighbors=90, min_dist=0.0) - 40s
UMAP 14/16 (n_neighbors=90, min_dist=0.1) - 44s
UMAP 15/16 (n_neighbors=90, min_dist=0.5) - 48s
UMAP 16/16 (n_neighbors=90, min_dist=0.9) - 51s


In [11]:
from IPython.display import display
import pandas as pd

pd.set_option('display.max_rows', None)      # affiche les 144 lignes
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

display(grid_results.round(2))

,current,n_neighbors,min_dist,min_cluster_size,min_samples,ARI_seeds,n_clusters_mean,n_clusters_min,n_clusters_max,trustworthiness,noise_pct,DBCV
0,,15,0.0,10,10,1.00,4.0,4,4,0.98,0.00,0.39
1,,15,0.0,15,10,1.00,4.0,4,4,0.98,0.00,0.39
2,,15,0.0,20,10,1.00,4.0,4,4,0.98,0.00,0.39
3,,15,0.0,20,1,0.96,4.2,4,5,0.98,0.00,0.57
4,,15,0.0,20,5,0.96,4.2,4,5,0.98,0.00,0.52
5,,50,0.0,20,5,0.95,3.2,3,4,0.99,0.79,0.26
6,,50,0.0,15,5,0.91,3.4,3,5,0.99,1.08,0.25
7,,30,0.0,20,1,0.90,5.0,4,6,0.99,0.72,0.42
8,,15,0.0,15,5,0.90,4.6,4,6,0.98,0.65,0.53
9,,15,0.1,15,10,0.90,4.0,4,4,0.99,4.62,0.20


In [ ]:
from pathlib import Path

grid_out = Path.home() / "HayLabAnalysis" / "Graph clustering"
grid_out.mkdir(parents=True, exist_ok=True)
grid_results.to_csv(grid_out / f"{grid_NrSubtype}_{grid_DataType}_grille_parametres.csv", index=False)